# Steam Sales Dataset Analysis (Baseline Modeling & Evaluation)

This section focuses on building and evaluating baseline machine learning models to predict game pricing based on textual and structured features derived from the Steam dataset. The primary objective is to establish a performance benchmark using relatively simple and interpretable models before moving on to more advanced techniques.

In this phase, the problem is framed as a regression task, where the goal is to estimate a game’s price from its description and associated metadata.

The baseline modeling workflow includes:

- Feature Extraction – Converting game descriptions into numerical vectors using methods like Count Vectorization
- Model Training – Applying regression algorithms such as Linear Regression and Random Forest Regressor
- Performance Evaluation – Measuring model accuracy using metrics like Mean Squared Error (MSE) and R² score
- Benchmarking – Establishing a reference point for future model improvements and comparisons

By implementing these baseline models, the analysis provides an initial understanding of how well pricing can be predicted from available data and highlights the strengths and limitations of traditional machine learning approaches in this context. This serves as a foundation for future enhancements, including more sophisticated models, feature engineering, and deep learning techniques.

## Import Libraries

In [1]:
import random
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sales_util.evaluator import evaluate
from sales_util.items_data import Item

In [2]:
# Load env file
load_dotenv(override=True)

True

## Load Dataset From HuggingFace

In [3]:
username = "KumudithaSilva"
dataset = f"{username}/items_llm_raw_lite"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")

README.md: 0.00B [00:00, ?B/s]

c:\Users\REDTECH\miniconda3\envs\ml-dl-fine-tuning\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\REDTECH\.cache\huggingface\hub\datasets--KumudithaSilva--items_llm_raw_lite. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/181k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/182k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Loaded 22,000 items


In [ ]:
# import importlib
# from sales_util import evaluator

# importlib.reload(evaluator)

### Error (Mean Error)
- It tells **how far off predictions are on average**
- The difference between the predicted value and the real value.
- Smaller error = better prediction.

---

### MSE (Mean Squared Error)
- Measures **how wrong predictions are, on average**
- It squares the errors before averaging them.
- Bigger mistakes are punished more.
- Lower MSE = better model.

---

### R² (R-squared)
- Shows **how well the model explains the data**
- Value can be:
  - **1** → perfect prediction
  - **0** → same as guessing the average
  - **negative** → very bad model (worse than guessing)

| Technique | Error   | MSE | R²        |
|----------|---------|-----|-----------|
| Random   | $20.05  | 582 | -1268.9%  |
| Linear Regression   | $3.95  | 38 | 11.0%  |
| Random Forest  | $4.05  | 35 | 17.4%  |
| XGB | $4.05  | 35 | 17.4%  |

## Random Predictor

In [4]:
def random_pricer(item):
    return random.randrange(1,49)

In [5]:
random.seed(42)
evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$28 $7 $2 $42 $12 $16 $2 $9 $4 $46 $18 $43 $40 $30 $4 $37 $21 $4 $3 $5 $1 $9 $33 $37 $1 $32 $10 $45 $41 $44 $26 $26 $13 $14 $36 $15 $4 $3 $44 $25 $18 $18 $7 $11 $20 $5 $1 $16 $4 $21 $14 $32 $16 $2 $32 $23 $31 $5 $25 $5 $26 $17 $39 $20 $22 $35 $10 $40 $4 $0 $43 $12 $10 $2 $11 $6 $24 $17 $27 $34 $14 $10 $23 $22 $13 $36 $17 $41 $43 $37 $2 $27 $40 $6 $34 $44 $15 $10 $30 $22 $17 $41 $41 $27 $3 $42 $19 $3 $5 $6 $18 $11 $12 $3 $1 $36 $34 $18 $7 $42 $31 $25 $32 $21 $9 $12 $9 $15 $33 $26 $5 $15 $42 $33 $23 $18 $21 $22 $5 $8 $20 $31 $5 $0 $7 $6 $36 $1 $38 $0 $37 $8 $20 $22 $37 $20 $32 $12 $30 $2 $43 $35 $7 $34 $33 $17 $40 $20 $6 $11 $22 $9 $27 $0 $42 $39 $7 $32 $10 $18 $6 $36 $2 $31 $32 $38 $10 $9 $18 $4 $28 $31 $1 $38 $21 $28 $2 $17 $21 $19 

## Linear Regression

In [6]:
def get_item_features(item):
    return {
        "peakCCU": item.peakCCU,
        "required_age": item.required_age,
        "dlc": item.dlcCount,
        "supportWindows": int(item.supportWindows),
        "supportMac": int(item.supportMac),
        "supportLinux": int(item.supportLinux),
        "positive": item.positive,
        "negative": item.negative,
        "achievements": item.achievements,
        "recommendation": item.recommendations or 0,
        "release_year": item.release_year or 0,
        "release_month": item.release_month or 0,
        "release_day": item.release_day or 0,
        "min_estimatedOwners": item.min_estimatedOwners or 0,
        "max_estimatedOwners": item.max_estimatedOwners or 0,
        "supported_languages": item.supported_languages or 0,
        "num_developers": item.num_developers or 0,
        "num_publishers": item.num_publishers or 0,
        "num_categories": item.num_categories or 0,
        "num_genres": item.num_genres or 0,
    }

In [7]:
get_item_features(items[213])

{'peakCCU': 0,
 'required_age': 0,
 'dlc': 0,
 'supportWindows': 1,
 'supportMac': 0,
 'supportLinux': 0,
 'positive': 3,
 'negative': 0,
 'achievements': 0,
 'recommendation': 0,
 'release_year': 2023,
 'release_month': 1,
 'release_day': 19,
 'min_estimatedOwners': 0,
 'max_estimatedOwners': 20000,
 'supported_languages': 3,
 'num_developers': 1,
 'num_publishers': 1,
 'num_categories': 3,
 'num_genres': 2}

In [8]:
def list_to_df(items):
    features = [get_item_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

In [9]:
train_df = list_to_df(train)
test_df = list_to_df(test)

In [10]:
# Linear Regression

np.random.seed(42)

feature_columns = train_df.columns.drop('price')

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [11]:
for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

peakCCU: 0.0005251212837058012
required_age: 0.10583562107010493
dlc: 0.3442021820529364
supportWindows: 3.5727677002134968
supportMac: 0.46165683875808877
supportLinux: -1.014248339015782
positive: 7.970847057831693e-05
negative: 0.00035272415139242394
achievements: -0.0004901169499439656
recommendation: -2.5445509870181992e-05
release_year: 0.1544722676968991
release_month: 0.03840115265126239
release_day: 0.020241848960292862
min_estimatedOwners: -4.897108481412188e-07
max_estimatedOwners: 2.0643670173017402e-07
supported_languages: -0.02983842064552183
num_developers: 0.23455692776705195
num_publishers: 0.18368200486085626
num_categories: 0.19797136839420526
num_genres: 0.07288384232767024
Intercept: -312.71408310251115


In [12]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [13]:
print(f"MSE: {mse}")
print("R²:", r2)

MSE: 29.804927979405225
R²: 0.05584556914684713


In [14]:
def linear_regression_pricer(item):
    features = get_item_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [15]:
evaluate(linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$8 $3 $4 $2 $2 $3 $9 $0 $1 $3 $20 $5 $1 $1 $6 $4 $2 $0 $1 $4 $10 $0 $3 $3 $4 $0 $2 $2 $2 $3 $4 $4 $3 $10 $3 $3 $1 $2 $5 $1 $8 $4 $2 $3 $4 $3 $0 $6 $2 $3 $3 $1 $4 $3 $9 $2 $2 $25 $4 $5 $4 $3 $3 $7 $4 $3 $3 $1 $4 $2 $4 $2 $3 $1 $1 $4 $4 $4 $2 $2 $5 $4 $4 $4 $4 $2 $7 $1 $3 $0 $4 $8 $3 $0 $6 $2 $4 $3 $4 $2 $3 $5 $1 $5 $7 $2 $2 $6 $4 $4 $2 $9 $0 $3 $10 $3 $4 $4 $3 $2 $0 $4 $4 $5 $3 $0 $4 $3 $11 $4 $26 $3 $0 $0 $1 $15 $1 $2 $5 $4 $8 $4 $4 $1 $3 $1 $1 $5 $1 $19 $2 $4 $1 $0 $3 $5 $3 $0 $1 $2 $3 $6 $4 $6 $2 $3 $2 $3 $2 $24 $47 $2 $2 $5 $0 $2 $5 $5 $3 $10 $6 $1 $9 $4 $4 $5 $1 $3 $1 $10 $2 $2 $5 $4 $5 $1 $2 $22 $1 $4 

## CountVector

In [16]:
prices = np.array([float(item.price) for item in train])
description = [item.small_description for item in train]

In [17]:
np.random.seed(42)
vectorize = CountVectorizer(max_features=2000, stop_words='english')
X = vectorize.fit_transform(description)

In [18]:
selected_words = vectorize.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

Number of selected words: 2000
Selected words: ['knights' 'knowledge' 'lab' 'labyrinth' 'labyrinths' 'land' 'landmarks'
 'lands' 'landscape' 'landscapes' 'language' 'laser' 'lasers' 'late'
 'launch' 'lava' 'lead' 'leaderboards' 'learn' 'learning']


## Linear Regression With CountVector

In [19]:
regressor = LinearRegression()
regressor.fit(X, prices)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [20]:
def natural_language_linear_regression_pricer(item):
    x = vectorize.transform([item.small_description])
    return max(regressor.predict(x)[0], 0)

In [21]:
evaluate(natural_language_linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$5 $7 $4 $5 $0 $6 $11 $0 $1 $3 $17 $3 $1 $1 $1 $1 $1 $1 $1 $3 $11 $4 $1 $2 $2 $3 $1 $4 $0 $8 $2 $5 $2 $4 $0 $3 $1 $1 $3 $0 $3 $4 $0 $1 $2 $3 $3 $2 $9 $8 $2 $1 $3 $1 $6 $1 $0 $2 $11 $4 $3 $6 $4 $16 $3 $0 $1 $1 $4 $4 $1 $4 $3 $2 $0 $0 $4 $1 $2 $3 $4 $2 $2 $4 $5 $0 $3 $2 $2 $2 $1 $8 $4 $0 $4 $4 $6 $1 $2 $1 $4 $3 $6 $7 $3 $3 $1 $2 $0 $2 $1 $2 $3 $3 $10 $6 $5 $1 $2 $0 $3 $7 $3 $4 $2 $0 $4 $5 $12 $6 $23 $2 $3 $4 $4 $17 $2 $4 $5 $4 $9 $3 $5 $0 $4 $1 $0 $6 $2 $22 $4 $9 $1 $1 $6 $4 $2 $0 $3 $3 $4 $6 $0 $7 $1 $5 $2 $5 $0 $17 $44 $6 $3 $4 $0 $2 $4 $8 $3 $6 $6 $3 $5 $2 $5 $6 $0 $1 $1 $8 $1 $1 $4 $5 $3 $4 $3 $19 $1 $6 

## Random Forest

In [22]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=4)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

In [23]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [24]:
print("MSE:", mse)
print("R²:", r2)

MSE: 26.81571398881122
R²: 0.15053728039465553


In [25]:
def random_forest_pricer(item):
    features = get_item_features(item)
    features_df = pd.DataFrame([features])
    return rf_model.predict(features_df)[0]

In [26]:
evaluate(random_forest_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$8 $3 $5 $2 $3 $3 $6 $2 $1 $1 $18 $3 $1 $1 $6 $2 $3 $8 $2 $4 $10 $5 $5 $2 $3 $1 $2 $3 $3 $2 $4 $2 $3 $7 $5 $4 $0 $4 $6 $3 $6 $2 $0 $3 $3 $3 $1 $4 $3 $4 $4 $0 $2 $3 $6 $2 $0 $13 $4 $3 $3 $3 $2 $6 $2 $3 $1 $3 $4 $1 $3 $1 $3 $1 $1 $3 $1 $8 $1 $2 $7 $1 $5 $4 $4 $0 $6 $1 $3 $1 $3 $8 $7 $1 $6 $0 $2 $3 $5 $7 $2 $6 $0 $3 $1 $2 $8 $5 $2 $4 $1 $11 $0 $6 $11 $5 $4 $5 $2 $4 $6 $4 $3 $6 $3 $1 $3 $2 $10 $4 $26 $3 $2 $0 $2 $13 $0 $2 $2 $3 $7 $1 $3 $1 $7 $0 $3 $6 $1 $16 $4 $4 $3 $2 $4 $1 $1 $1 $4 $8 $2 $6 $12 $4 $2 $3 $2 $5 $1 $22 $36 $17 $1 $4 $0 $7 $5 $7 $4 $8 $6 $1 $2 $0 $6 $2 $1 $1 $2 $8 $4 $2 $5 $2 $12 $3 $1 $1 $0 $5 

## XGBoost

In [27]:
xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

In [28]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [29]:
print("MSE:", mse)
print("R²:", r2)

MSE: 28.009481444179727
R²: 0.11272135837084518


In [30]:
def xgb_pricer(item):
    features = get_item_features(item)
    features_df = pd.DataFrame([features])
    return rf_model.predict(features_df)[0]

In [31]:
evaluate(xgb_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$8 $3 $5 $2 $3 $3 $6 $2 $1 $1 $18 $3 $1 $1 $6 $2 $3 $8 $2 $4 $10 $5 $5 $2 $3 $1 $2 $3 $3 $2 $4 $2 $3 $7 $5 $4 $0 $4 $6 $3 $6 $2 $0 $3 $3 $3 $1 $4 $3 $4 $4 $0 $2 $3 $6 $2 $0 $13 $4 $3 $3 $3 $2 $6 $2 $3 $1 $3 $4 $1 $3 $1 $3 $1 $1 $3 $1 $8 $1 $2 $7 $1 $5 $4 $4 $0 $6 $1 $3 $1 $3 $8 $7 $1 $6 $0 $2 $3 $5 $7 $2 $6 $0 $3 $1 $2 $8 $5 $2 $4 $1 $11 $0 $6 $11 $5 $4 $5 $2 $4 $6 $4 $3 $6 $3 $1 $3 $2 $10 $4 $26 $3 $2 $0 $2 $13 $0 $2 $2 $3 $7 $1 $3 $1 $7 $0 $3 $6 $1 $16 $4 $4 $3 $2 $4 $1 $1 $1 $4 $8 $2 $6 $12 $4 $2 $3 $2 $5 $1 $22 $36 $17 $1 $4 $0 $7 $5 $7 $4 $8 $6 $1 $2 $0 $6 $2 $1 $1 $2 $8 $4 $2 $5 $2 $12 $3 $1 $1 $0 $5 